# 02 · Data Quality Assessment

Applies the five core data-quality dimensions — **completeness, validity, consistency, accuracy,
timeliness** — to the synthetic submissions generated in `01_data_generation.ipynb`, using
standard statistical techniques:

- Missingness pattern analysis (visual matrix + a Little's-MCAR-style diagnostic)
- Statistical process control (p-chart) for timeliness / rejection monitoring
- Pareto analysis of validation-rule violations
- Cohen's kappa for accuracy agreement on an audited sample

The notebook ends by exporting a static HTML dashboard page (`docs/dashboard/data-quality.html`).

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from pathlib import Path
from scipy import stats

DATA_DIR = Path("../data")
DOCS_DIR = Path("../docs")

districts = pd.read_csv(DATA_DIR / "districts.csv")
submissions = pd.read_csv(DATA_DIR / "submissions.csv")
violations = pd.read_csv(DATA_DIR / "validation_violations.csv")
rules = pd.read_csv(DATA_DIR / "validation_rules.csv")

PLOT_TEMPLATE = "plotly_white"
COLORWAY = ["#12A594", "#16213E", "#E8A33D", "#D64550", "#566089", "#2FC4B2"]
print(f"{len(submissions):,} submission records loaded across {submissions.collection.nunique()} collections.")

11,520 submission records loaded across 8 collections.


## 1. Completeness

In [2]:
overall_completeness = submissions.completeness_rate.mean()
by_collection = submissions.groupby("collection").completeness_rate.mean().sort_values()
by_region = submissions.groupby("region").completeness_rate.mean().sort_values()
trend = submissions.groupby("period").completeness_rate.mean().reset_index()

print(f"Statewide mean completeness: {overall_completeness:.1%}")
by_collection.round(3)

Statewide mean completeness: 93.0%


collection
SIRS Special Education        0.930
SIRS Assessment               0.930
IDEA 618 Exiting              0.930
SIRS Enrollment               0.931
IDEA Maintenance of Effort    0.931
IDEA 618 Child Count          0.931
Personnel Master File         0.931
Student Attendance (EMSC)     0.931
Name: completeness_rate, dtype: float64

In [3]:
fig = make_subplots(rows=1, cols=2, subplot_titles=("Completeness by Collection", "Monthly Trend (statewide)"))
fig.add_trace(go.Bar(y=by_collection.index, x=by_collection.values, orientation="h",
                      marker_color="#12A594", name="Completeness"), row=1, col=1)
fig.add_trace(go.Scatter(x=trend.period, y=trend.completeness_rate, mode="lines+markers",
                          line=dict(color="#16213E", width=2), name="Statewide"), row=1, col=2)
fig.add_hline(y=0.98, line_dash="dot", line_color="#D64550", row=1, col=2,
              annotation_text="98% target")
fig.update_layout(template=PLOT_TEMPLATE, showlegend=False, height=420,
                   title="Completeness Rate — Collection Mix & Trend")
fig.update_xaxes(tickformat=".0%", row=1, col=1)
fig.update_yaxes(tickformat=".0%", row=1, col=2)
fig.show()

### Missingness pattern diagnostic

Completeness at the *collection* level hides *which fields* are driving it. Here we simulate a raw field-level extract for a sample of submission rows and inspect the missingness pattern directly — this is the step that would normally precede a full Little's MCAR test on a real extract.

In [4]:
rng = np.random.default_rng(11)
sample = submissions.sample(400, random_state=11).reset_index(drop=True).copy()

# Simulate field-level nulls whose *rate* depends on the row's own completeness_rate
# (i.e., missingness is NOT completely at random — it's related to an observed variable,
#  which is the textbook definition of Missing At Random, MAR).
fields = ["disability_code", "iep_date", "exit_reason", "primary_language", "case_manager_id"]
extract = pd.DataFrame({"row_completeness": sample.completeness_rate})
for f in fields:
    miss_prob = np.clip(1 - sample.completeness_rate + rng.normal(0, 0.05, len(sample)), 0.01, 0.9)
    extract[f] = np.where(rng.random(len(sample)) < miss_prob, np.nan, 1.0)

missing_matrix = extract[fields].isna().astype(int)

fig = go.Figure(go.Heatmap(
    z=missing_matrix.T.values, x=list(range(len(extract))), y=fields,
    colorscale=[[0, "#ECEEF3"], [1, "#D64550"]], showscale=False,
))
fig.update_layout(template=PLOT_TEMPLATE, height=320,
                   title="Field-level missingness matrix (400-row sample; red = missing)",
                   xaxis_title="Record index", yaxis_title=None)
fig.show()

# --- simplified MCAR-style diagnostic ---
# For each field, test whether the DISTRIBUTION of an observed covariate (row_completeness)
# differs between the missing and non-missing groups. If it differs significantly for most
# fields, missingness is systematically related to an observed variable -> not MCAR (i.e. MAR).
print("Per-field t-test: row_completeness differs between missing vs. present group?\n")
mcar_flag = True
for f in fields:
    grp_missing = extract.loc[missing_matrix[f] == 1, "row_completeness"]
    grp_present = extract.loc[missing_matrix[f] == 0, "row_completeness"]
    t_stat, p_val = stats.ttest_ind(grp_missing, grp_present, equal_var=False)
    verdict = "differs (→ suggests MAR)" if p_val < 0.05 else "no significant difference"
    if p_val < 0.05:
        mcar_flag = False
    print(f"  {f:20s}  t={t_stat:6.2f}  p={p_val:7.4f}   {verdict}")

print(f"\nOverall diagnostic: {'consistent with MCAR' if mcar_flag else 'NOT consistent with MCAR (missingness is Missing-At-Random)'}")

Per-field t-test: row_completeness differs between missing vs. present group?

  disability_code       t= -4.92  p= 0.0000   differs (→ suggests MAR)
  iep_date              t= -4.23  p= 0.0002   differs (→ suggests MAR)
  exit_reason           t= -2.88  p= 0.0076   differs (→ suggests MAR)
  primary_language      t= -4.88  p= 0.0000   differs (→ suggests MAR)
  case_manager_id       t= -3.79  p= 0.0008   differs (→ suggests MAR)

Overall diagnostic: NOT consistent with MCAR (missingness is Missing-At-Random)


## 2. Validity — error rates & rule-violation Pareto

In [5]:
error_trend = submissions.groupby("period").validity_error_rate.mean().reset_index()
rule_totals = violations.groupby(["rule_id", "description", "severity"]).violation_count.sum() \
                         .reset_index().sort_values("violation_count", ascending=False)
rule_totals["cum_pct"] = rule_totals.violation_count.cumsum() / rule_totals.violation_count.sum()

fig = make_subplots(specs=[[{"secondary_y": True}]])
fig.add_trace(go.Bar(x=rule_totals.rule_id, y=rule_totals.violation_count,
                      marker_color=np.where(rule_totals.severity == "Error", "#D64550", "#E8A33D"),
                      name="Violations"))
fig.add_trace(go.Scatter(x=rule_totals.rule_id, y=rule_totals.cum_pct, mode="lines+markers",
                          line=dict(color="#16213E", width=2), name="Cumulative %"), secondary_y=True)
fig.add_hline(y=0.8, line_dash="dot", line_color="#566089", secondary_y=True,
              annotation_text="80% line")
fig.update_layout(template=PLOT_TEMPLATE, height=440,
                   title="Pareto of Validation Rule Violations (statewide, both school years)")
fig.update_yaxes(title_text="Violation count", secondary_y=False)
fig.update_yaxes(title_text="Cumulative share", tickformat=".0%", secondary_y=True, range=[0, 1.05])
fig.show()

top3 = rule_totals.head(3)
print(f"Top 3 rules account for {top3.violation_count.sum() / rule_totals.violation_count.sum():.0%} of all violations:")
top3[["rule_id", "description", "violation_count"]]

Top 3 rules account for 54% of all violations:


,rule_id,description,violation_count
0,R001,Required field missing,1436
11,R012,Record submitted after collection window closed,870
4,R005,SWD flag set but no IEP date on file,745


## 3. Consistency — cross-field & referential checks

In [6]:
checks = []

# Referential integrity: every submission district_id exists in the district master file
orphan_districts = set(submissions.district_id) - set(districts.district_id)
checks.append(("Referential: district_id exists in master file", len(orphan_districts) == 0,
                f"{len(orphan_districts)} orphan district_id(s)"))

# Logical bound: submitted records shouldn't meaningfully exceed expected
over_submitted = (submissions.records_submitted > submissions.records_expected * 1.05).sum()
checks.append(("Consistency: records_submitted <= 105% of records_expected", over_submitted == 0,
                f"{over_submitted} rows exceed bound ({over_submitted / len(submissions):.2%})"))

# SWD count cannot exceed total enrollment
bad_swd = (districts.swd_count > districts.enrollment_total).sum()
checks.append(("Consistency: swd_count <= enrollment_total", bad_swd == 0,
                f"{bad_swd} district(s) violate"))

# Completeness rate implied by submitted/expected should match the recorded completeness_rate
implied = (submissions.records_submitted / submissions.records_expected).round(2)
mismatch = (implied != submissions.completeness_rate.round(2)).sum()
checks.append(("Consistency: completeness_rate matches submitted/expected ratio",
                mismatch / len(submissions) < 0.01, f"{mismatch} mismatches ({mismatch/len(submissions):.2%})"))

check_df = pd.DataFrame(checks, columns=["check", "passed", "detail"])
check_df["status"] = check_df.passed.map({True: "PASS", False: "REVIEW"})
check_df[["check", "status", "detail"]]

,check,status,detail
0,Referential: district_id exists in master file,PASS,0 orphan district_id(s)
1,Consistency: records_submitted <= 105% of reco...,PASS,0 rows exceed bound (0.00%)
2,Consistency: swd_count <= enrollment_total,PASS,0 district(s) violate
3,Consistency: completeness_rate matches submitt...,REVIEW,3262 mismatches (28.32%)


## 4. Accuracy — Cohen's kappa on the audited sample

For the ~10% of district-collection-months that received a manual accuracy audit, we simulate a second, independent auditor rating (with realistic inter-rater noise) and measure agreement using Cohen's kappa — the standard statistic for categorical inter-rater reliability.

In [7]:
from sklearn.metrics import cohen_kappa_score, confusion_matrix

audited = submissions.dropna(subset=["accuracy_score"]).copy()
audited["system_label"] = np.where(audited.accuracy_score >= 0.90, "Accurate", "Inaccurate")

rng = np.random.default_rng(5)
# second rater agrees most of the time but has independent noise -> realistic kappa < 1
noisy_score = np.clip(audited.accuracy_score.values + rng.normal(0, 0.06, len(audited)), 0, 1)
audited["auditor_label"] = np.where(noisy_score >= 0.90, "Accurate", "Inaccurate")

kappa = cohen_kappa_score(audited.system_label, audited.auditor_label)
cm = confusion_matrix(audited.system_label, audited.auditor_label, labels=["Accurate", "Inaccurate"])

print(f"Audited records: {len(audited):,}")
print(f"Cohen's kappa (system vs. auditor): {kappa:.3f}")
interp = ("almost perfect" if kappa > 0.8 else "substantial" if kappa > 0.6 else
          "moderate" if kappa > 0.4 else "fair" if kappa > 0.2 else "slight")
print(f"Interpretation (Landis & Koch, 1977): {interp} agreement")

fig = go.Figure(go.Heatmap(z=cm, x=["Auditor: Accurate", "Auditor: Inaccurate"],
                            y=["System: Accurate", "System: Inaccurate"],
                            text=cm, texttemplate="%{text}", colorscale="Teal", showscale=False))
fig.update_layout(template=PLOT_TEMPLATE, height=360,
                   title=f"System vs. Auditor Agreement — Cohen's κ = {kappa:.3f}")
fig.show()

Audited records: 1,189
Cohen's kappa (system vs. auditor): 0.659
Interpretation (Landis & Koch, 1977): substantial agreement


## 5. Timeliness — p-chart (statistical process control)

In [8]:
monthly = submissions.groupby("period").agg(n=("on_time", "size"), late=("on_time", lambda s: (~s).sum()))
monthly["p"] = monthly.late / monthly.n
p_bar = monthly.late.sum() / monthly.n.sum()
monthly["ucl"] = p_bar + 3 * np.sqrt(p_bar * (1 - p_bar) / monthly.n)
monthly["lcl"] = (p_bar - 3 * np.sqrt(p_bar * (1 - p_bar) / monthly.n)).clip(lower=0)
monthly = monthly.reset_index()
monthly["out_of_control"] = monthly.p > monthly.ucl

fig = go.Figure()
fig.add_trace(go.Scatter(x=monthly.period, y=monthly.p, mode="lines+markers", name="Late rate (p)",
                          line=dict(color="#16213E", width=2),
                          marker=dict(color=np.where(monthly.out_of_control, "#D64550", "#16213E"), size=8)))
fig.add_trace(go.Scatter(x=monthly.period, y=monthly.ucl, mode="lines", name="UCL (3σ)",
                          line=dict(color="#D64550", dash="dot")))
fig.add_trace(go.Scatter(x=monthly.period, y=[p_bar]*len(monthly), mode="lines", name="Center line",
                          line=dict(color="#9AA6C3", dash="dash")))
fig.add_trace(go.Scatter(x=monthly.period, y=monthly.lcl, mode="lines", name="LCL (3σ)",
                          line=dict(color="#12A594", dash="dot")))
fig.update_layout(template=PLOT_TEMPLATE, height=420, yaxis_tickformat=".1%",
                   title="p-Chart — Statewide Late-Submission Rate by Month",
                   xaxis_title="Reporting month", yaxis_title="Proportion late")
fig.show()

n_oos = monthly.out_of_control.sum()
print(f"Center line p̄ = {p_bar:.2%}. Months exceeding the 3σ upper control limit: {n_oos}")

Center line p̄ = 9.44%. Months exceeding the 3σ upper control limit: 0


## 6. Quality summary table (exported for the dashboard)

In [9]:
quality_summary = submissions.groupby("collection").agg(
    mean_completeness=("completeness_rate", "mean"),
    mean_validity_error=("validity_error_rate", "mean"),
    on_time_rate=("on_time", "mean"),
    mean_duplicate_count=("duplicate_count", "mean"),
    n_records=("district_id", "size"),
).reset_index().sort_values("mean_completeness")
quality_summary.to_csv(DATA_DIR / "quality_summary.csv", index=False)
quality_summary.round(3)

,collection,mean_completeness,mean_validity_error,on_time_rate,mean_duplicate_count,n_records
6,SIRS Special Education,0.930,0.055,0.899,0.578,1440
4,SIRS Assessment,0.930,0.052,0.919,0.579,1440
1,IDEA 618 Exiting,0.930,0.054,0.905,0.547,1440
5,SIRS Enrollment,0.931,0.053,0.902,0.567,1440
2,IDEA Maintenance of Effort,0.931,0.053,0.906,0.548,1440
0,IDEA 618 Child Count,0.931,0.053,0.911,0.576,1440
3,Personnel Master File,0.931,0.053,0.895,0.563,1440
7,Student Attendance (EMSC),0.931,0.055,0.907,0.589,1440


## 7. Export static dashboard page

In [10]:
import json
NAV_PAGES = json.load(open("../config/nav_pages.json"))

def render_nav(active):
    links = "\n".join(
        f'<a href="{href}" class="{"active" if key == active else ""}">{label}</a>'
        for key, label, href in NAV_PAGES
    )
    return f'''<nav class="site-nav"><div class="nav-inner">
      <div class="brand"><span class="dot"></span> NYSED Data Integrity Platform</div>
      <div class="links">{links}</div>
    </div></nav>'''

def page_shell(title, eyebrow, description, body_html, active, filename):
    html = f'''<!DOCTYPE html>
<html lang="en"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>{title} · NYSED Data Integrity Platform</title>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link href="https://fonts.googleapis.com/css2?family=Lora:wght@500;600;700&family=Inter:wght@400;500;600;700&family=JetBrains+Mono:wght@400;600;700&display=swap" rel="stylesheet">
<link rel="stylesheet" href="../assets/style.css">
</head><body>
{render_nav(active)}
<header class="dash-header"><div class="dash-header-inner">
  <div class="eyebrow">{eyebrow}</div>
  <h1>{title}</h1>
  <p>{description}</p>
</div></header>
<main class="dash-body">
{body_html}
</main>
<footer class="site-footer">
  Synthetic demonstration data · NYSED Data Integrity &amp; Risk Monitoring Platform ·
  <a href="https://github.com/zia207/NYSED-Data-Integrity" target="_blank">Source code</a>
</footer>
</body></html>'''
    out = DOCS_DIR / "dashboard" / filename
    out.write_text(html)
    print("wrote", out)

def kpi_card(label, value, delta=None, tone="good", warn=False, crit=False):
    cls = "kpi-card" + (" crit" if crit else " warn" if warn else "")
    delta_html = f'<div class="kpi-delta {tone}">{delta}</div>' if delta else ""
    return f'''<div class="{cls}"><div class="kpi-label">{label}</div>
    <div class="kpi-value">{value}</div>{delta_html}</div>'''

In [11]:
kpi_html = f'''<div class="kpi-row">
  {kpi_card("Statewide Completeness", f"{overall_completeness:.1%}", "Target ≥ 98%",
             tone="good" if overall_completeness>=0.98 else "bad", warn=overall_completeness<0.98)}
  {kpi_card("Mean Validity Error Rate", f"{submissions.validity_error_rate.mean():.1%}", "Target ≤ 2%",
             tone="bad" if submissions.validity_error_rate.mean()>0.02 else "good",
             warn=submissions.validity_error_rate.mean()>0.02)}
  {kpi_card("On-Time Submission Rate", f"{submissions.on_time.mean():.1%}", "Target ≥ 95%",
             tone="good" if submissions.on_time.mean()>=0.95 else "bad", warn=submissions.on_time.mean()<0.95)}
  {kpi_card("Cohen&#39;s κ (Accuracy Audit)", f"{kappa:.2f}", interp.title())}
</div>'''

chart_html = f'''
<div class="chart-panel">
  <h3>Completeness — by collection &amp; monthly trend</h3>
  <div class="chart-note">Statewide mean completeness is {overall_completeness:.1%}; the 98% institutional target is marked on the trend line.</div>
  {fig.to_html(include_plotlyjs=False, full_html=False, div_id="completeness-fig") if False else ""}
</div>
'''

# Re-render each figure to embedded HTML divs (first occurrence of plotly.js only)
import plotly.io as pio

def fig_div(f, div_id, include_js=False):
    return pio.to_html(f, include_plotlyjs="inline" if include_js else False, full_html=False, div_id=div_id)

# Rebuild the figures we want on the page (kept as variables from cells above: reuse by recomputing quickly)
fig_completeness = make_subplots(rows=1, cols=2, subplot_titles=("Completeness by Collection", "Monthly Trend (statewide)"))
fig_completeness.add_trace(go.Bar(y=by_collection.index, x=by_collection.values, orientation="h",
                      marker_color="#12A594", name="Completeness"), row=1, col=1)
fig_completeness.add_trace(go.Scatter(x=trend.period, y=trend.completeness_rate, mode="lines+markers",
                          line=dict(color="#16213E", width=2), name="Statewide"), row=1, col=2)
fig_completeness.add_hline(y=0.98, line_dash="dot", line_color="#D64550", row=1, col=2)
fig_completeness.update_layout(template=PLOT_TEMPLATE, showlegend=False, height=380, margin=dict(t=40,l=10,r=10,b=10))
fig_completeness.update_xaxes(tickformat=".0%", row=1, col=1)
fig_completeness.update_yaxes(tickformat=".0%", row=1, col=2)

fig_pareto = make_subplots(specs=[[{"secondary_y": True}]])
fig_pareto.add_trace(go.Bar(x=rule_totals.rule_id, y=rule_totals.violation_count,
                      marker_color=np.where(rule_totals.severity == "Error", "#D64550", "#E8A33D"), name="Violations"))
fig_pareto.add_trace(go.Scatter(x=rule_totals.rule_id, y=rule_totals.cum_pct, mode="lines+markers",
                          line=dict(color="#16213E", width=2), name="Cumulative %"), secondary_y=True)
fig_pareto.add_hline(y=0.8, line_dash="dot", line_color="#566089", secondary_y=True)
fig_pareto.update_layout(template=PLOT_TEMPLATE, height=380, margin=dict(t=30,l=10,r=10,b=10))
fig_pareto.update_yaxes(title_text="Violations", secondary_y=False)
fig_pareto.update_yaxes(title_text="Cumulative", tickformat=".0%", secondary_y=True, range=[0, 1.05])

fig_pchart = go.Figure()
fig_pchart.add_trace(go.Scatter(x=monthly.period, y=monthly.p, mode="lines+markers", name="Late rate",
                          line=dict(color="#16213E", width=2),
                          marker=dict(color=np.where(monthly.out_of_control, "#D64550", "#16213E"), size=7)))
fig_pchart.add_trace(go.Scatter(x=monthly.period, y=monthly.ucl, mode="lines", name="UCL", line=dict(color="#D64550", dash="dot")))
fig_pchart.add_trace(go.Scatter(x=monthly.period, y=monthly.lcl, mode="lines", name="LCL", line=dict(color="#12A594", dash="dot")))
fig_pchart.update_layout(template=PLOT_TEMPLATE, height=380, yaxis_tickformat=".1%", margin=dict(t=30,l=10,r=10,b=10))

fig_missing = go.Figure(go.Heatmap(z=missing_matrix.T.values, x=list(range(len(extract))), y=fields,
                                    colorscale=[[0, "#ECEEF3"], [1, "#D64550"]], showscale=False))
fig_missing.update_layout(template=PLOT_TEMPLATE, height=300, margin=dict(t=20,l=10,r=10,b=10))

body = f'''
{kpi_html}
<div class="chart-panel">
  <h3>Completeness — by collection &amp; monthly trend</h3>
  <div class="chart-note">Statewide mean completeness is {overall_completeness:.1%} against a 98% target (dotted line).</div>
  {fig_div(fig_completeness, "fig-completeness", include_js=True)}
</div>
<div class="chart-row">
  <div class="chart-panel">
    <h3>Field-level missingness pattern</h3>
    <div class="chart-note">400-row sample; missingness rate is related to overall row completeness → consistent with <b>MAR</b>, not MCAR.</div>
    {fig_div(fig_missing, "fig-missing")}
  </div>
  <div class="chart-panel">
    <h3>Timeliness p-chart</h3>
    <div class="chart-note">3σ control limits around the statewide late-submission rate (p̄ = {p_bar:.1%}).</div>
    {fig_div(fig_pchart, "fig-pchart")}
  </div>
</div>
<div class="chart-panel">
  <h3>Validation rule violations — Pareto</h3>
  <div class="chart-note">Top 3 rules ({", ".join(top3.rule_id)}) account for {top3.violation_count.sum() / rule_totals.violation_count.sum():.0%} of all violations.</div>
  {fig_div(fig_pareto, "fig-pareto")}
</div>
<div class="chart-panel">
  <h3>Consistency checks</h3>
  <table class="kpi-table"><thead><tr><th>Check</th><th>Status</th><th>Detail</th></tr></thead><tbody>
  {''.join(f"<tr><td>{r.check}</td><td><span class='pill pill-quality'>{r.status}</span></td><td class='target'>{r.detail}</td></tr>" for r in check_df.itertuples())}
  </tbody></table>
</div>
'''

page_shell(
    title="Data Quality",
    eyebrow="Notebook 02",
    description="Completeness, validity, consistency, accuracy, and timeliness — measured with missingness diagnostics, Pareto analysis, Cohen's kappa, and a statistical-process-control p-chart.",
    body_html=body, active="data-quality.html", filename="data-quality.html",
)

wrote ../docs/dashboard/data-quality.html
